In [ ]:
from md_Helpers import (
    CavitationConfig,
    ProjectPaths,
    SQLiteRunDatabase,
    run_cavitation,
)

paths = ProjectPaths()
database = SQLiteRunDatabase(paths.database)
database.initialize()

# Only select known homogeneous thermalization states.
thermalizations = database.query_thermalizations(
    Phase_Separation_Status="Not_Separated",
    Pressure_Mean=(0, .2),
    This_LJ_Time=(400, None),
    Clone_Run_ID=None,
    Therm_kT = 1.00,
    Nsteps = 200_000,
    N_Cells = (31,None)
)

source_run_ids = [row["Run_ID"] for row in thermalizations]

print(len(source_run_ids))
source_run_ids

In [ ]:
results = []

for source_run_id in source_run_ids:
    config = CavitationConfig(
        source_run_id=source_run_id,
        mask_radius=15.0,
        nsteps=500_000,
        ensemble="NVT",
        random_location=False,
    )

    try:
        result = run_cavitation(
            config,
            project_paths=paths,
            database=database,
        )
    except Exception as error:
        result = {
            "source_run_id": source_run_id,
            "status": "Failed",
            "error": f"{type(error).__name__}: {error}",
        }

    results.append(result)
    print(source_run_id, result.get("status"), result.get("run_id"))